In [3]:
# !pip install requests beautifulsoup4 pandas openpyxl -q

import os
import re
import json
import time
import random
from collections import deque
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from urllib.parse import urlparse, urljoin, urldefrag

import requests
import pandas as pd
from bs4 import BeautifulSoup

# =========================================================
# CONFIG
# =========================================================
BASE_URL = "https://tireex.com"
START_PAGES = [
    BASE_URL + "/",             # главная
    # если знаешь раздел каталога — добавь сюда (ускорит и улучшит coverage):
    # BASE_URL + "/shop/",
    # BASE_URL + "/tires/",
    # BASE_URL + "/tyres/",
]

OUT_XLSX = "tireex_tyres.xlsx"
URLCACHE_PATH = "tireex_all_product_urls.json"
CHECKPOINT_PATH = "tireex_checkpoint.json"

# Сеть/скорость
SLEEP = 0.35
TIMEOUT = 40
MAX_RETRIES = 6

# Автосохранение результатов
SAVE_EVERY_ROWS = 200

# Автосохранение кеша discovery
SAVE_CACHE_EVERY_PAGES = 200

# Ограничения (можно оставить None)
TEST_LIMIT = None                 # например 500 для теста; None=без лимита
MAX_CRAWL_PAGES = None            # например 20000 для страховки; None=без лимита (риск долго)
MAX_QUEUE = 500000                # защита от бесконечного разрастания очереди ссылок

# Логи
VERBOSE_ITEMS = False             # печатать каждый товар (лучше False для больших объёмов)
PRINT_PROGRESS_EVERY = 200        # печать прогресса каждые N страниц/товаров

# =========================================================
# SESSION
# =========================================================
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,ru;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Connection": "keep-alive",
})

# =========================================================
# HELPERS
# =========================================================
def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", (x or "")).strip()

def normalize_url(u: str) -> str:
    """
    - to absolute
    - remove #fragment
    - drop obvious tracking query params
    - keep query when it looks like pagination (?page=2)
    """
    if not u:
        return ""
    u = urljoin(BASE_URL, u)
    u, _frag = urldefrag(u)

    p = urlparse(u)
    # only same domain
    if p.netloc and p.netloc != urlparse(BASE_URL).netloc:
        return ""

    # drop file types
    if (p.path or "").lower().endswith((".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg", ".pdf", ".zip", ".xml", ".mp4")):
        return ""

    # normalize query: keep only pagination-like keys if any
    if p.query:
        # keep page/p/paged only; drop others to reduce duplicates
        qs = p.query
        keep = []
        for part in qs.split("&"):
            k = part.split("=", 1)[0].lower().strip()
            if k in ("page", "p", "paged"):
                keep.append(part)
        if keep:
            u = p._replace(query="&".join(keep)).geturl()
        else:
            u = p._replace(query="").geturl()

    return u

def safe_decimal(x) -> Decimal | None:
    if x is None:
        return None
    if isinstance(x, (int, float, Decimal)):
        try:
            return Decimal(str(x)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        except InvalidOperation:
            return None
    s = clean_text(str(x)).replace(",", ".")
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m:
        return None
    try:
        return Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    except InvalidOperation:
        return None

def normalize_size_195_65R15(text: str) -> str | None:
    t = clean_text(text)
    m = re.search(r"\b(\d{3})\s*[/\-]\s*(\d{2,3})\s*R\s*(\d{2})\b", t, flags=re.I)
    if m:
        return f"{m.group(1)}/{m.group(2)}R{m.group(3)}"
    m2 = re.search(r"\b(\d{3})\s+(\d{2,3})\s*R\s*(\d{2})\b", t, flags=re.I)
    if m2:
        return f"{m2.group(1)}/{m2.group(2)}R{m2.group(3)}"
    return None

def request_with_retries(url: str) -> requests.Response:
    last_err = None
    for i in range(1, MAX_RETRIES + 1):
        try:
            r = SESSION.get(url, timeout=TIMEOUT, allow_redirects=True)
            return r
        except requests.RequestException as e:
            last_err = e
            sleep_s = min(15, (1.2 ** i) + random.uniform(0.3, 1.2))
            print(f"NET ERR retry {i}/{MAX_RETRIES} -> sleep {sleep_s:.2f}s | {type(e).__name__}: {e}")
            time.sleep(sleep_s)
    raise last_err

def get_soup(url: str) -> BeautifulSoup:
    r = request_with_retries(url)
    print(f"    GET {url} -> {r.status_code}")
    r.raise_for_status()
    time.sleep(SLEEP)
    return BeautifulSoup(r.text, "html.parser")

def access_blocked(status: int | None) -> bool:
    return status in (403, 429)

# =========================================================
# CHECKPOINT / CACHE IO
# =========================================================
def load_json(path: str, default):
    if not os.path.exists(path):
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

def save_json(path: str, data):
    tmp = path + ".__tmp__"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def load_checkpoint() -> dict:
    """
    Универсальный checkpoint:
    {
      "mode": "crawl"|"parse"|"done",
      "crawl": {"queue": [...], "seen_pages": [...], "product_urls": [...], "pages_done": N},
      "parse": {"next_index": K, "rows": M},
      "ts": ...
    }
    """
    return load_json(CHECKPOINT_PATH, {
        "mode": "crawl",
        "crawl": {"queue": START_PAGES, "seen_pages": [], "product_urls": [], "pages_done": 0},
        "parse": {"next_index": 0, "rows": 0},
        "ts": time.time(),
    })

def save_checkpoint(cp: dict):
    cp["ts"] = time.time()
    save_json(CHECKPOINT_PATH, cp)
    print(f"✅ Checkpoint -> {os.path.abspath(CHECKPOINT_PATH)} | mode={cp.get('mode')}")

def load_existing_xlsx(out_path: str) -> tuple[list[dict], set[str]]:
    if not os.path.exists(out_path):
        return [], set()
    try:
        df = pd.read_excel(out_path, sheet_name="Data")
        if df is None or df.empty or "url" not in df.columns:
            return [], set()
        df = df.dropna(subset=["url"]).copy()
        df["url"] = df["url"].astype(str).str.strip()
        seen = set(df["url"].tolist())
        rows = df.to_dict(orient="records")
        print(f"📌 Existing file loaded: {os.path.basename(out_path)} | rows={len(rows)} | seen_urls={len(seen)}")
        return rows, seen
    except Exception as e:
        print(f"⚠️ Can't read existing {out_path}: {e}")
        return [], set()

# =========================================================
# PIVOT + SAVE (атомарно)
# =========================================================
def build_pivot_min_price(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    needed = {"size", "brand", "price"}
    if not needed.issubset(df_raw.columns):
        return pd.DataFrame()

    df = df_raw.copy()
    df["Price"] = pd.to_numeric(df["price"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["Size"] = df["size"].astype(str).str.strip()
    df["Brand"] = df["brand"].astype(str).str.strip()

    df = df.dropna(subset=["Price"]).copy()
    df = df[(df["Size"] != "") & (df["Brand"] != "")]
    pivot = df.pivot_table(index="Size", columns="Brand", values="Price", aggfunc="min").round(2)
    return pivot.sort_index()

def save_all(out_path: str, rows: list[dict]):
    df = pd.DataFrame(rows)
    pivot = build_pivot_min_price(df)

    tmp_path = out_path + ".__tmp__.xlsx"
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name="Data")
        pivot.to_excel(w, sheet_name="Pivot")

    os.replace(tmp_path, out_path)
    st = os.stat(out_path)
    uniq = df["url"].nunique() if ("url" in df.columns and len(df) > 0) else 0
    print(f"💾 SAVE -> {os.path.abspath(out_path)} | rows={len(df)} | uniq_url={uniq} | mtime={time.ctime(st.st_mtime)}")

# =========================================================
# DISCOVERY: CRAWL SITE -> product_urls
# =========================================================
def looks_like_listing_url(u: str) -> bool:
    """
    Решаем, стоит ли добавлять URL в очередь обхода.
    """
    p = urlparse(u)
    path = (p.path or "").lower()

    # отсекаем явно не нужное
    if any(x in path for x in ["/cart", "/checkout", "/account", "/login", "/register", "/wp-admin", "/admin", "/privacy", "/terms"]):
        return False

    # обычно листинги: /shop /category /product-category /tires /tyres
    if any(x in path for x in ["/shop", "/category", "/categories", "/collection", "/collections", "/tires", "/tyres", "/products"]):
        return True

    # пагинация через query (page/p/paged уже оставили в normalize_url)
    if p.query and any(k in p.query.lower() for k in ["page=", "p=", "paged="]):
        return True

    # главная/короткие страницы тоже ок как “хабы”
    if path.strip("/") in ("", "home"):
        return True

    # иначе — не раздуваем очередь всем подряд
    return False

def looks_like_product_url(u: str) -> bool:
    """
    Быстрый URL-фильтр. Не идеально, но помогает.
    """
    p = urlparse(u)
    path = (p.path or "").lower()

    # типовые товарные пути
    if any(x in path for x in ["/product/", "/products/", "/tyre/", "/tyres/", "/tire/", "/tires/"]):
        # отсечём бренды/категории
        if any(x in path for x in ["/brand", "/brands", "/category", "/categories", "/collection", "/collections", "/filter"]):
            return False
        # чаще всего у товара slug из 2+ сегментов
        return len(path.strip("/").split("/")) >= 2

    return False

def page_has_product_signals(soup: BeautifulSoup) -> bool:
    """
    Сигналы карточки товара:
    - JSON-LD Product
    - наличие price / amount / meta itemprop=price
    """
    # JSON-LD
    for s in soup.select('script[type="application/ld+json"]'):
        txt = (s.string or s.get_text() or "").strip()
        if not txt:
            continue
        if '"@type"' in txt and "Product" in txt:
            return True

    # meta itemprop price
    if soup.select_one('[itemprop="price"][content], meta[property="product:price:amount"][content]'):
        return True

    # видимые price блоки (очень грубо, но полезно)
    price_candidates = soup.select(".price, .amount, .product-price, [class*='price']")
    if price_candidates:
        t = " ".join(clean_text(x.get_text(" ", strip=True)) for x in price_candidates[:8])
        if safe_decimal(t) is not None:
            return True

    return False

def extract_links(soup: BeautifulSoup) -> list[str]:
    links = []
    for a in soup.select("a[href]"):
        href = a.get("href")
        u = normalize_url(href)
        if not u:
            continue
        # same domain only already ensured
        links.append(u)
    # dedupe keep order
    out = []
    seen = set()
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def run_discovery(cp: dict) -> dict:
    crawl = cp.get("crawl", {})
    queue = deque(crawl.get("queue", START_PAGES))
    seen_pages = set(crawl.get("seen_pages", []))
    product_urls = set(crawl.get("product_urls", []))
    pages_done = int(crawl.get("pages_done", 0))

    print("\n=== DISCOVERY (crawl) ===")
    print("queue:", len(queue), "| seen_pages:", len(seen_pages), "| product_urls:", len(product_urls), "| pages_done:", pages_done)

    while queue:
        if MAX_CRAWL_PAGES is not None and pages_done >= MAX_CRAWL_PAGES:
            print(f"⏹️ Reached MAX_CRAWL_PAGES={MAX_CRAWL_PAGES}. Stop discovery.")
            break

        if len(queue) > MAX_QUEUE:
            print(f"⏹️ Queue too large ({len(queue)}). Stop to avoid runaway. Add better START_PAGES.")
            break

        url = queue.popleft()
        if url in seen_pages:
            continue
        seen_pages.add(url)

        try:
            soup = get_soup(url)
        except requests.HTTPError as e:
            status = getattr(e.response, "status_code", None)
            if access_blocked(status):
                print("⚠️ Access blocked (403/429) during discovery. Saving progress and exit to resume later...")
                # save checkpoint with current state and stop
                cp["mode"] = "crawl"
                cp["crawl"] = {
                    "queue": list(queue),
                    "seen_pages": list(seen_pages),
                    "product_urls": list(product_urls),
                    "pages_done": pages_done,
                }
                save_checkpoint(cp)
                raise
            # 404/прочие — игнорим
            pass
        except Exception as e:
            # сетевые сбои — сохранимся и выйдем, чтобы продолжить
            print(f"⚠️ Discovery error on {url}: {type(e).__name__}: {e}")
            cp["mode"] = "crawl"
            cp["crawl"] = {
                "queue": list(queue),
                "seen_pages": list(seen_pages),
                "product_urls": list(product_urls),
                "pages_done": pages_done,
            }
            save_checkpoint(cp)
            raise

        pages_done += 1

        # если URL выглядит как товар — добавим кандидатом
        if looks_like_product_url(url):
            product_urls.add(url)
        else:
            # если URL не выглядит как товар, но страница содержит явные сигналы товара — тоже добавим
            # (так ловим магазины без /product/)
            try:
                if page_has_product_signals(soup):
                    product_urls.add(url)
            except Exception:
                pass

        # достаём ссылки и добавляем в очередь (только “хабы/листинги” + возможные товары)
        try:
            links = extract_links(soup)
            for u in links:
                if u in seen_pages:
                    continue
                if looks_like_product_url(u):
                    product_urls.add(u)
                    # товарные можно не класть в очередь discovery (чтобы не раздувать),
                    # но иногда товар содержит ссылки на вариации — поэтому добавим, но с фильтром:
                    queue.append(u)
                else:
                    if looks_like_listing_url(u):
                        queue.append(u)
        except Exception:
            pass

        # прогресс
        if pages_done % PRINT_PROGRESS_EVERY == 0:
            print(f"--- discovery progress: pages_done={pages_done} | seen_pages={len(seen_pages)} | product_urls={len(product_urls)} | queue={len(queue)} ---")

        # периодически сохраняем cache + checkpoint
        if pages_done % SAVE_CACHE_EVERY_PAGES == 0:
            cache_payload = {
                "base": BASE_URL,
                "ts": time.time(),
                "product_urls": sorted(product_urls),
                "seen_pages": len(seen_pages),
                "pages_done": pages_done,
                "note": "discovered by crawl (no sitemap)",
            }
            save_json(URLCACHE_PATH, cache_payload)
            print(f"✅ URLs cached -> {URLCACHE_PATH} | product_urls={len(product_urls)}")

            cp["mode"] = "crawl"
            cp["crawl"] = {
                "queue": list(queue),
                "seen_pages": list(seen_pages),
                "product_urls": list(product_urls),
                "pages_done": pages_done,
            }
            save_checkpoint(cp)

    # финальное сохранение cache
    cache_payload = {
        "base": BASE_URL,
        "ts": time.time(),
        "product_urls": sorted(product_urls),
        "seen_pages": len(seen_pages),
        "pages_done": pages_done,
        "note": "discovered by crawl (no sitemap)",
    }
    save_json(URLCACHE_PATH, cache_payload)
    print(f"✅ URLs cached -> {URLCACHE_PATH} | product_urls={len(product_urls)}")

    cp["mode"] = "parse"
    cp["crawl"] = {
        "queue": list(queue),
        "seen_pages": list(seen_pages),
        "product_urls": list(product_urls),
        "pages_done": pages_done,
    }
    save_checkpoint(cp)
    return cp

# =========================================================
# PRODUCT PARSER (универсальный: JSON-LD -> meta -> common selectors)
# =========================================================
def extract_jsonld_products(soup: BeautifulSoup) -> list[dict]:
    items = []
    for s in soup.select('script[type="application/ld+json"]'):
        txt = (s.string or s.get_text() or "").strip()
        if not txt:
            continue
        try:
            data = json.loads(txt)
        except Exception:
            continue

        def walk(obj):
            if isinstance(obj, dict):
                t = obj.get("@type") or obj.get("type")
                if isinstance(t, list):
                    t_list = [str(x).lower() for x in t]
                else:
                    t_list = [str(t).lower()] if t else []
                if "product" in t_list or str(t).lower() == "product":
                    items.append(obj)
                for v in obj.values():
                    walk(v)
            elif isinstance(obj, list):
                for x in obj:
                    walk(x)
        walk(data)
    return items

def parse_product_page(url: str) -> dict | None:
    soup = get_soup(url)

    brand = None
    model = None
    price = None
    old_price = None
    currency = None
    size = None
    year = None

    # 1) JSON-LD
    prods = extract_jsonld_products(soup)
    if prods:
        p = prods[0]
        model = p.get("name") or model
        b = p.get("brand")
        if isinstance(b, dict):
            brand = b.get("name") or brand
        elif isinstance(b, str):
            brand = b

        offers = p.get("offers")
        if isinstance(offers, dict):
            price = safe_decimal(offers.get("price"))
            currency = offers.get("priceCurrency") or currency
        elif isinstance(offers, list) and offers:
            price = safe_decimal(offers[0].get("price"))
            currency = offers[0].get("priceCurrency") or currency

    # 2) h1/title
    if not model:
        h1 = soup.select_one("h1")
        if h1:
            model = clean_text(h1.get_text(" ", strip=True))
    if not model:
        if soup.title:
            model = clean_text(soup.title.get_text(" ", strip=True))

    # size from model/title
    if model:
        size = normalize_size_195_65R15(model) or size
    if not size and soup.title:
        size = normalize_size_195_65R15(soup.title.get_text(" ", strip=True)) or size

    # 3) price meta/itemprop
    if price is None:
        meta_price = soup.select_one('[itemprop="price"][content]')
        if meta_price and meta_price.get("content"):
            price = safe_decimal(meta_price["content"])
    if currency is None:
        meta_cur = soup.select_one('[itemprop="priceCurrency"][content]')
        if meta_cur and meta_cur.get("content"):
            currency = clean_text(meta_cur["content"])

    # 4) common price selectors (woo/shopify-like)
    if price is None:
        candidates = soup.select(
            ".price .amount, .price ins .amount, .price, .product-price, .product__price, "
            ".woocommerce-Price-amount, [class*='price'] .amount, [class*='price']"
        )
        for c in candidates[:25]:
            t = clean_text(c.get_text(" ", strip=True))
            d = safe_decimal(t)
            if d is not None:
                price = d
                break

    # old price
    if old_price is None:
        node_old = soup.select_one("del .amount, .old-price, .was-price, .price del")
        if node_old:
            old_price = safe_decimal(clean_text(node_old.get_text(" ", strip=True)))

    # brand fallback
    if not brand:
        og_brand = soup.select_one('meta[property="product:brand"][content], meta[name="brand"][content]')
        if og_brand and og_brand.get("content"):
            brand = clean_text(og_brand["content"])

    if not brand:
        txt = soup.get_text("\n", strip=True)
        m = re.search(r"\bBrand\b\s*[:\-]?\s*([A-Za-z0-9][A-Za-z0-9 \-]{1,40})", txt, flags=re.I)
        if m:
            brand = clean_text(m.group(1))

    # year heuristic
    txt2 = soup.get_text(" ", strip=True)
    m_year = re.search(r"\b(20\d{2})\b", txt2)
    if m_year:
        y = int(m_year.group(1))
        if 2015 <= y <= 2035:
            year = y

    # ONLY актуальные товары с ценой
    if price is None:
        return None

    row = {
        "size": size or "",
        "brand": clean_text(brand or "UNKNOWN"),
        "model": clean_text(model or ""),
        "year": year,
        "price": str(price),
        "old_price": str(old_price) if old_price is not None else None,
        "currency": currency or "",
        "url": url,
    }
    return row

# =========================================================
# MAIN
# =========================================================
print("\n=== TIREEX PARSER (pitstop-style: crawl cache + checkpoint + seen_urls) ===")
print("CWD:", os.getcwd())
print("OUT:", os.path.abspath(OUT_XLSX))
print("CHK:", os.path.abspath(CHECKPOINT_PATH))
print("URLCACHE:", os.path.abspath(URLCACHE_PATH))

cp = load_checkpoint()

# 1) if cache exists and mode already parse -> skip discovery
cache = load_json(URLCACHE_PATH, {})
cached_urls = cache.get("product_urls") if isinstance(cache, dict) else None
if isinstance(cached_urls, list) and cached_urls:
    product_urls = cached_urls
    print(f"✅ URLs cache loaded: {URLCACHE_PATH} | product_urls={len(product_urls)}")
    # если checkpoint был crawl — переведём в parse
    if cp.get("mode") == "crawl":
        cp["mode"] = "parse"
        save_checkpoint(cp)
else:
    # discovery
    if cp.get("mode") != "crawl":
        # если checkpoint сломан — начинаем discovery
        cp["mode"] = "crawl"
        cp["crawl"] = {"queue": START_PAGES, "seen_pages": [], "product_urls": [], "pages_done": 0}
        save_checkpoint(cp)
    cp = run_discovery(cp)
    cache = load_json(URLCACHE_PATH, {})
    product_urls = cache.get("product_urls", [])
    print(f"✅ Discovery done. product_urls={len(product_urls)}")

print("\nВсего URL товаров к обработке:", len(product_urls))

# 2) load existing
all_rows, seen_urls = load_existing_xlsx(OUT_XLSX)

# 3) resume parse index
next_index = int(cp.get("parse", {}).get("next_index", 0))
if next_index < 0:
    next_index = 0
if next_index > len(product_urls):
    next_index = len(product_urls)

print(f"Resume: next_index={next_index} | already_rows={len(all_rows)} | seen_urls={len(seen_urls)}")

added_since_save = 0
processed = 0
sk_404 = 0
sk_noprice = 0
sk_other = 0

try:
    for idx in range(next_index, len(product_urls)):
        url = product_urls[idx]

        # dedupe
        if url in seen_urls:
            continue

        # test limit
        if TEST_LIMIT is not None and len(all_rows) >= TEST_LIMIT:
            print("⏹️ TEST_LIMIT reached")
            save_all(OUT_XLSX, all_rows)
            cp["mode"] = "parse"
            cp["parse"] = {"next_index": idx, "rows": len(all_rows)}
            save_checkpoint(cp)
            break

        try:
            row = parse_product_page(url)
            processed += 1

            if row is None:
                sk_noprice += 1
            else:
                seen_urls.add(url)
                all_rows.append(row)
                added_since_save += 1

                if VERBOSE_ITEMS:
                    print(f"    OK: {row['brand']} | {row.get('size','')} | {row.get('model','')[:60]} | year:{row.get('year')} | price:{row.get('price')} | {url}")

        except requests.HTTPError as e:
            status = getattr(e.response, "status_code", None)
            if status == 404:
                sk_404 += 1
            else:
                print(f"\n❌ HTTPError on URL: {url}\n   {type(e).__name__}: {e}")
                if access_blocked(status):
                    print("⚠️ Access blocked (403/429). Saving progress and exit to resume later...")
                    save_all(OUT_XLSX, all_rows)
                    cp["mode"] = "parse"
                    cp["parse"] = {"next_index": idx, "rows": len(all_rows)}
                    save_checkpoint(cp)
                    raise
                sk_other += 1

        except Exception as e:
            print(f"\n❌ Error on URL: {url}\n   {type(e).__name__}: {e}")
            sk_other += 1

        # progress
        if (idx + 1) % PRINT_PROGRESS_EVERY == 0:
            print(f"--- progress: idx={idx+1}/{len(product_urls)} | processed={processed} | rows={len(all_rows)} | added_since_save={added_since_save} | uniq={len(seen_urls)} ---")

        # autosave
        if added_since_save >= SAVE_EVERY_ROWS:
            save_all(OUT_XLSX, all_rows)
            cp["mode"] = "parse"
            cp["parse"] = {"next_index": idx + 1, "rows": len(all_rows)}
            save_checkpoint(cp)
            added_since_save = 0

    # final save
    save_all(OUT_XLSX, all_rows)
    cp["mode"] = "done"
    cp["parse"] = {"next_index": len(product_urls), "rows": len(all_rows)}
    save_checkpoint(cp)

except Exception:
    # аварийный сейв (на всякий)
    try:
        save_all(OUT_XLSX, all_rows)
        cp["mode"] = "parse"
        # next_index уже сохранён выше перед raise при 403/429; если нет — хотя бы текущий
        cp["parse"] = {"next_index": int(cp.get("parse", {}).get("next_index", next_index)), "rows": len(all_rows)}
        save_checkpoint(cp)
    except Exception:
        pass
    raise

print("\n========================================")
df_final = pd.DataFrame(all_rows)
uniq = df_final["url"].nunique() if ("url" in df_final.columns and len(df_final) > 0) else 0
print(f"✅ DONE. rows={len(df_final)} | uniq_url={uniq}")
print(f"skipped: 404={sk_404} | no_price_or_not_product={sk_noprice} | other={sk_other}")
print(f"✅ File: {os.path.abspath(OUT_XLSX)} (sheets: Data, Pivot)")
print(f"✅ Cache: {os.path.abspath(URLCACHE_PATH)}")
print(f"✅ Checkpoint: {os.path.abspath(CHECKPOINT_PATH)}")



=== TIREEX PARSER (pitstop-style: crawl cache + checkpoint + seen_urls) ===
CWD: C:\Users\mochkasov
OUT: C:\Users\mochkasov\tireex_tyres.xlsx
CHK: C:\Users\mochkasov\tireex_checkpoint.json
URLCACHE: C:\Users\mochkasov\tireex_all_product_urls.json
✅ URLs cache loaded: tireex_all_product_urls.json | product_urls=825

Всего URL товаров к обработке: 825
Resume: next_index=0 | already_rows=0 | seen_urls=0
    GET https://tireex.com -> 200
    GET https://tireex.com/ -> 200
    GET https://tireex.com/en/product/arisun-175-65-r14-82h-2/ -> 200
    GET https://tireex.com/en/product/arisun-175-70-r14-84t-2/ -> 200
    GET https://tireex.com/en/product/arisun-195-55-55-r15-85v-2/ -> 200
    GET https://tireex.com/en/product/arisun-195-55-r15-85v/ -> 202
    GET https://tireex.com/en/product/arisun-195-65-r15-91h-2/ -> 200
    GET https://tireex.com/en/product/arisun-195-r15-tl-2/ -> 200
    GET https://tireex.com/en/product/arisun-205-55-r16-91v-tl/ -> 200
    GET https://tireex.com/en/product/

In [19]:
# !pip install requests beautifulsoup4 pandas openpyxl -q

import os
import re
import json
import time
import random
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

# =========================
# CONFIG
# =========================
BASE = "https://tireex.com"
START_URLS = [
    "https://tireex.com/en/",
    "https://tireex.com/en/shop/",
]

OUT_XLSX = "tireex_tyres.xlsx"
CHECKPOINT_PATH = "tireex_checkpoint.json"
URLCACHE_PATH = "tireex_all_product_urls.json"

SAVE_EVERY_ADDED = 100
SLEEP_BASE = 0.35
SLEEP_JITTER = 0.25
TIMEOUT = 40
MAX_RETRIES = 6

DISCOVERY_MAX_PAGES = 20000
VERBOSE = True

# Поставь True на ОДИН запуск, если хочешь начать с нуля (удалит xlsx/cache/checkpoint)
RESET_RUN = False

# =========================
# HTTP SESSION
# =========================
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

def _sleep():
    time.sleep(SLEEP_BASE + random.random() * SLEEP_JITTER)

def request_with_retries(url: str) -> requests.Response:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = SESSION.get(url, timeout=TIMEOUT, allow_redirects=True)
            return r
        except Exception as e:
            last_err = e
            backoff = min(20.0, 0.8 * (attempt ** 1.35) + random.random())
            print(f"NET ERR retry {attempt}/{MAX_RETRIES} -> sleep {backoff:.2f}s | {type(e).__name__}: {e}")
            time.sleep(backoff)
    raise last_err

def get_soup(url: str) -> tuple[BeautifulSoup, int]:
    r = request_with_retries(url)
    code = r.status_code
    print(f"    GET {url} -> {code}")
    _sleep()
    return BeautifulSoup(r.text or "", "html.parser"), code

# =========================
# UTILS
# =========================
def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", (x or "")).strip()

def is_http_url(u: str) -> bool:
    try:
        p = urlparse(u)
        return p.scheme in ("http", "https")
    except Exception:
        return False

def normalize_en_product_url(u: str) -> str | None:
    if not u:
        return None
    u = u.strip()
    if u.startswith("//"):
        u = "https:" + u
    if not is_http_url(u):
        return None

    p = urlparse(u)
    if p.netloc and "tireex.com" not in p.netloc:
        return None

    path = p.path or ""
    if "/product/" not in path:
        return None

    if path.startswith("/product/"):
        path = "/en" + path

    if not path.startswith("/en/product/"):
        m = re.search(r"/product/.*", path)
        if not m:
            return None
        path = "/en" + m.group(0)

    norm = f"{BASE}{path}"
    if not norm.endswith("/"):
        norm += "/"
    return norm

def should_crawl_page(u: str) -> bool:
    if not u or "tireex.com" not in u:
        return False
    p = urlparse(u)
    path = (p.path or "").lower()

    if any(x in path for x in ["/wp-json", "/cart", "/checkout", "/my-account", "/wishlist", "/compare"]):
        return False
    if any(path.endswith(ext) for ext in [".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg", ".pdf", ".zip"]):
        return False

    if path.startswith("/en/shop") or path.startswith("/en/product-category/"):
        return True
    if re.search(r"^/en/shop/page/\d+/?$", path):
        return True
    return False

def parse_size_from_text(text: str) -> str | None:
    t = clean_text(text)
    m = re.search(r"(\d{3})\s*[\/-]\s*(\d{2,3})\s*R\s*(\d{2})", t, flags=re.I)
    if m:
        return f"{m.group(1)}/{m.group(2)}R{m.group(3)}"
    return None

def parse_size_from_url(url: str) -> str | None:
    m = re.search(r"-(\d{3})-(\d{2,3})-r(\d{2})(?:-|/)", url, flags=re.I)
    if m:
        return f"{m.group(1)}/{m.group(2)}R{m.group(3)}"
    return None

def brand_from_title(title: str) -> str | None:
    t = clean_text(title)
    if not t:
        return None
    m = re.search(r"\b\d{3}\s*[\/-]\s*\d{2,3}\s*R\s*\d{2}\b", t, flags=re.I)
    if not m:
        b = t.split(" ")[0].strip()
        return b if b else None
    b = t[:m.start()].strip(" -|")
    b = clean_text(b)
    b = re.sub(r"\btires\b", "", b, flags=re.I).strip()
    b = clean_text(b)
    return b or None

def parse_specs_kv(soup: BeautifulSoup) -> dict[str, str]:
    out = {}
    ul = soup.select_one("ul.product-specifications-list")
    if not ul:
        return out
    for li in ul.select("li"):
        kp = li.select_one("p")
        vh = li.select_one("h6")
        k = clean_text(kp.get_text(" ", strip=True)) if kp else ""
        v = clean_text(vh.get_text(" ", strip=True)) if vh else ""
        if k and v:
            out[k.lower()] = v
    return out

def parse_price_currency(soup: BeautifulSoup) -> tuple[Decimal | None, str | None, Decimal | None]:
    tw = soup.select_one("tamara-widget[amount]")
    if tw and tw.get("amount"):
        raw = tw["amount"].strip()
        try:
            d = Decimal(raw).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        except InvalidOperation:
            d = None
        return d, "SAR", None

    meta_amt = soup.select_one('meta[property="product:price:amount"][content]')
    meta_cur = soup.select_one('meta[property="product:price:currency"][content]')
    if meta_amt and meta_amt.get("content"):
        try:
            d = Decimal(meta_amt["content"].strip()).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        except InvalidOperation:
            d = None
        cur = meta_cur["content"].strip().upper() if (meta_cur and meta_cur.get("content")) else "SAR"
        return d, cur, None

    price_node = soup.select_one(".price .woocommerce-Price-amount bdi, .summary .price bdi, p.price bdi")
    if price_node:
        t = clean_text(price_node.get_text(" ", strip=True)).replace(",", "")
        m = re.search(r"(\d+(?:\.\d+)?)", t)
        d = None
        if m:
            try:
                d = Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            except InvalidOperation:
                d = None

        old = None
        old_node = soup.select_one(".price del .woocommerce-Price-amount bdi, .price del bdi")
        if old_node:
            tt = clean_text(old_node.get_text(" ", strip=True)).replace(",", "")
            mm = re.search(r"(\d+(?:\.\d+)?)", tt)
            if mm:
                try:
                    old = Decimal(mm.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
                except InvalidOperation:
                    old = None

        return d, "SAR", old

    return None, None, None

def is_product_page(soup: BeautifulSoup) -> bool:
    return bool(soup.select_one("h1.product_title")) or bool(soup.select_one("body.single-product"))

# =========================
# EXCEL
# =========================
def build_pivot_min_price(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    needed = {"size", "brand", "price"}
    if not needed.issubset(df_raw.columns):
        return pd.DataFrame()

    df = df_raw.copy()
    df["Price"] = pd.to_numeric(df["price"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["Size"] = df["size"].astype(str).str.strip()
    df["Brand"] = df["brand"].astype(str).str.strip()
    df = df.dropna(subset=["Price"]).copy()
    df = df[(df["Size"] != "") & (df["Brand"] != "")]
    pivot = df.pivot_table(index="Size", columns="Brand", values="Price", aggfunc="min").round(2)
    return pivot.sort_index()

def save_all_atomic(out_path: str, rows: list[dict]):
    df = pd.DataFrame(rows)
    pivot = build_pivot_min_price(df)

    tmp = out_path + ".__tmp__.xlsx"
    with pd.ExcelWriter(tmp, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name="Data")
        pivot.to_excel(w, sheet_name="Pivot")

    os.replace(tmp, out_path)
    st = os.stat(out_path)
    uniq = df["url"].nunique() if ("url" in df.columns and len(df) > 0) else 0
    print(f"💾 SAVE -> {os.path.abspath(out_path)} | rows={len(df)} | uniq_url={uniq} | mtime={time.ctime(st.st_mtime)}")

def load_existing_xlsx(out_path: str) -> tuple[list[dict], set[str]]:
    if not os.path.exists(out_path):
        return [], set()
    try:
        df = pd.read_excel(out_path, sheet_name="Data")
        if df is None or df.empty or "url" not in df.columns:
            return [], set()
        df = df.dropna(subset=["url"]).copy()
        df["url"] = df["url"].astype(str).str.strip()
        seen = set(df["url"].tolist())
        rows = df.to_dict(orient="records")
        print(f"📌 Existing file loaded: {out_path} | rows={len(rows)} | seen_urls={len(seen)}")
        return rows, seen
    except Exception as e:
        print(f"⚠️ Can't read existing xlsx {out_path}: {e}")
        return [], set()

# =========================
# CHECKPOINT + URL CACHE
# =========================
def save_checkpoint(mode: str, next_index: int, total_urls: int, rows_count: int):
    data = {
        "mode": mode,
        "next_index": int(next_index),
        "total_urls": int(total_urls),
        "rows_count": int(rows_count),
        "ts": time.time(),
    }
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"✅ Checkpoint -> {os.path.abspath(CHECKPOINT_PATH)} | mode={mode} | next_index={next_index} | rows={rows_count}")

def load_checkpoint() -> dict | None:
    if not os.path.exists(CHECKPOINT_PATH):
        return None
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def save_url_cache(path: str, urls: list[str]):
    payload = {"ts": time.time(), "product_urls": urls, "count": len(urls)}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"✅ URLs cached -> {os.path.abspath(path)} | product_urls={len(urls)}")

def load_url_cache(path: str) -> list[str]:
    if not os.path.exists(path):
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        urls = data.get("product_urls") or []
        urls = [u for u in urls if isinstance(u, str) and u.strip()]
        print(f"✅ URLs cache loaded: {os.path.abspath(path)} | product_urls={len(urls)}")
        return urls
    except Exception as e:
        print(f"⚠️ Can't load URL cache: {e}")
        return []

# =========================
# DISCOVERY
# =========================
def discover_product_urls(max_pages: int = DISCOVERY_MAX_PAGES) -> list[str]:
    queue = list(START_URLS)
    visited = set()
    products = set()
    pages = 0

    while queue and pages < max_pages:
        page_url = queue.pop(0)
        if not page_url:
            continue
        if page_url.startswith("/"):
            page_url = urljoin(BASE, page_url)
        if not is_http_url(page_url):
            continue

        pu = urlparse(page_url)
        key = f"{pu.scheme}://{pu.netloc}{pu.path}".rstrip("/")
        if key in visited:
            continue
        visited.add(key)

        soup, code = get_soup(page_url)
        pages += 1
        if code >= 400:
            continue

        for a in soup.select("a[href]"):
            href = (a.get("href") or "").strip()
            if not href:
                continue
            absu = urljoin(page_url, href)

            prod = normalize_en_product_url(absu)
            if prod:
                products.add(prod)
                continue

            if should_crawl_page(absu):
                queue.append(absu)

        nxt = soup.select_one('a[rel="next"][href]')
        if nxt and nxt.get("href"):
            nx = urljoin(page_url, nxt["href"])
            if should_crawl_page(nx):
                queue.append(nx)

        if pages % 20 == 0:
            print(f"--- discovery pages={pages} | products_found={len(products)} | queue={len(queue)} ---")

    return sorted(products)

# =========================
# PRODUCT PARSER
# =========================
def parse_product_page(url: str) -> dict | None:
    url = normalize_en_product_url(url) or url

    soup, code = get_soup(url)
    if code == 404:
        return None
    if code == 403:
        raise PermissionError("403 Forbidden (likely blocked / VPN issue)")

    if not is_product_page(soup):
        return None

    h1 = soup.select_one("h1.product_title")
    title = clean_text(h1.get_text(" ", strip=True)) if h1 else ""
    if not title:
        return None

    specs = parse_specs_kv(soup)

    year = None
    if "year of production" in specs:
        m = re.search(r"\b(20\d{2})\b", specs["year of production"])
        if m:
            year = int(m.group(1))

    model = clean_text(specs.get("pattern", ""))  # Pattern = модель

    brand = brand_from_title(title) or ""
    brand = re.sub(r"\btires\b", "", brand, flags=re.I).strip()
    brand = clean_text(brand)

    size = parse_size_from_text(title) or parse_size_from_url(url) or ""

    price, currency, old_price = parse_price_currency(soup)
    if price is None:
        return None  # только товары с ценой

    return {
        "size": size,
        "brand": brand,
        "model": model,
        "year": year,
        "price": str(price),
        "old_price": (str(old_price) if old_price is not None else ""),
        "currency": (currency or "SAR"),
        "url": url,
    }

# =========================
# MAIN
# =========================
print("\n=== TIREEX PARSER (pitstop-style: crawl cache + checkpoint + seen_urls) ===")
print("CWD:", os.getcwd())
print("OUT:", os.path.abspath(OUT_XLSX))
print("CHK:", os.path.abspath(CHECKPOINT_PATH))
print("URLCACHE:", os.path.abspath(URLCACHE_PATH))

if RESET_RUN:
    for p in [OUT_XLSX, CHECKPOINT_PATH, URLCACHE_PATH]:
        if os.path.exists(p):
            os.remove(p)
            print("🧹 removed:", p)

product_urls = load_url_cache(URLCACHE_PATH)
if not product_urls:
    print("🧭 No valid URL cache (or empty). Running discovery crawl…")
    product_urls = discover_product_urls()
    save_url_cache(URLCACHE_PATH, product_urls)

total_urls = len(product_urls)
print("\nВсего URL товаров к обработке:", total_urls)

all_rows, seen_urls = load_existing_xlsx(OUT_XLSX)

cp = load_checkpoint() or {}
next_index = int(cp.get("next_index", 0))
if next_index < 0:
    next_index = 0
if next_index > total_urls:
    next_index = 0

print(f"Resume: next_index={next_index} | already_rows={len(all_rows)} | seen_urls={len(seen_urls)}")

added_since_save = 0
processed = 0

sk_404 = 0
sk_403 = 0
sk_no_price_or_not_product = 0
sk_other = 0

try:
    for i in range(next_index, total_urls):
        url = product_urls[i]
        processed += 1

        if url in seen_urls:
            continue

        try:
            row = parse_product_page(url)
            if not row:
                sk_no_price_or_not_product += 1
                seen_urls.add(url)
                continue

            seen_urls.add(row["url"])
            all_rows.append(row)
            added_since_save += 1

            if VERBOSE:
                print(f"    OK: {row['brand']} | {row['size']} | {row['model']} | year: {row['year']} | price: {row['price']} {row['currency']} | {row['url']}")

            if added_since_save >= SAVE_EVERY_ADDED:
                save_all_atomic(OUT_XLSX, all_rows)
                save_checkpoint("parse", i + 1, total_urls, len(all_rows))  # <--- ВСЁ ПОЗИЦИОННО
                added_since_save = 0

            if processed % 200 == 0:
                print(f"--- progress: idx={i+1}/{total_urls} | processed={processed} | rows={len(all_rows)} | uniq={len(seen_urls)} ---")

        except PermissionError:
            sk_403 += 1
            print(f"\n❌ 403 on URL: {url}")
            print("⚠️ Saving progress and exiting. Fix access/VPN, then rerun cell to resume…")
            save_all_atomic(OUT_XLSX, all_rows)
            save_checkpoint("parse", i, total_urls, len(all_rows))  # продолжим с i
            raise

        except requests.HTTPError as he:
            if "404" in str(he):
                sk_404 += 1
            else:
                sk_other += 1

        except Exception as e:
            sk_other += 1
            print(f"\n❌ Error on URL: {url}\n   {type(e).__name__}: {e}")
            save_all_atomic(OUT_XLSX, all_rows)
            save_checkpoint("parse", i + 1, total_urls, len(all_rows))

    save_all_atomic(OUT_XLSX, all_rows)
    save_checkpoint("parse", total_urls, total_urls, len(all_rows))

except Exception:
    pass

print("\n========================================")
df_final = pd.DataFrame(all_rows)
uniq = df_final["url"].nunique() if ("url" in df_final.columns and len(df_final) > 0) else 0
print(f"✅ DONE. rows={len(df_final)} | uniq_url={uniq}")
print(f"skipped: 404={sk_404} | 403={sk_403} | no_price_or_not_product={sk_no_price_or_not_product} | other={sk_other}")
print(f"✅ File: {os.path.abspath(OUT_XLSX)} (sheets: Data, Pivot)")
print(f"✅ Checkpoint: {os.path.abspath(CHECKPOINT_PATH)}")
print(f"✅ URL cache: {os.path.abspath(URLCACHE_PATH)}")



=== TIREEX PARSER (pitstop-style: crawl cache + checkpoint + seen_urls) ===
CWD: C:\Users\mochkasov
OUT: C:\Users\mochkasov\tireex_tyres.xlsx
CHK: C:\Users\mochkasov\tireex_checkpoint.json
URLCACHE: C:\Users\mochkasov\tireex_all_product_urls.json
🧭 No valid URL cache (or empty). Running discovery crawl…
    GET https://tireex.com/en/ -> 200
    GET https://tireex.com/en/shop/ -> 200
    GET https://tireex.com/en/product-category/kumho-tires/ -> 200
    GET https://tireex.com/en/product-category/alpha-tires/ -> 200
    GET https://tireex.com/en/product-category/anchee-tires/ -> 200
    GET https://tireex.com/en/product-category/trazano-tires/ -> 200
    GET https://tireex.com/en/product-category/greentrac-tires/ -> 202
    GET https://tireex.com/en/product-category/goodride-tires/ -> 200
    GET https://tireex.com/en/product-category/giti-tires/ -> 200
    GET https://tireex.com/en/product-category/dynamo-tires/ -> 200
    GET https://tireex.com/en/product-category/doublestar-tires/ ->